# 02a — WorldPop Population Extraction (Large TIF / Low RAM)

**Designed for:** 750MB India TIF on a standard laptop (8–16GB RAM)  
**Key technique:** Windowed reading — only the Delhi rectangle is ever loaded into memory. The 750MB file stays on disk untouched.

**Input :** `data/raw/population.tif` (full India, 750MB, do NOT move or rename)  
**Outputs:**
- `data/processed/population_delhi.tif` — clipped Delhi raster (~2MB)
- `data/processed/grid_with_population.pkl` — 500m grid + population columns
- `data/processed/population_features.pkl` — numpy array for the ML pipeline

**Run order:** `01_grid_build` → **`02a_worldpop`** → `02_features` → `03_train` → `04_bip`

## 0. Install dependencies (run once, then restart kernel)

In [1]:
# Uncomment and run ONCE if not already installed
%pip install rasterio rasterstats geopandas shapely numpy pandas matplotlib --quiet
# Then restart your Jupyter kernel before continuing

Note: you may need to restart the kernel to use updated packages.


## 1. Imports

In [2]:
import os
import gc
import pickle
import warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

import rasterio
from rasterio.windows import from_bounds
from rasterio.warp    import reproject, Resampling, calculate_default_transform
from rasterstats       import zonal_stats
from shapely.geometry import box

warnings.filterwarnings('ignore')
print('All imports OK.')

All imports OK.


## 2. Configuration — must match 01_grid_build.ipynb exactly

In [3]:
# ── Bounding box — Delhi ───────────────────────────────────────────────
MIN_LON = 76.8
MIN_LAT = 28.4
MAX_LON = 77.4
MAX_LAT = 28.9

# ── File paths ─────────────────────────────────────────────────────────
TIF_IN       = 'data/raw/population.tif'
TIF_DELHI    = 'data/processed/population_delhi.tif'
GRID_PATH    = 'data/processed/grid.geojson'
OUT_GRID_PKL = 'data/processed/grid_with_population.pkl'
OUT_FEAT_PKL = 'data/processed/population_features.pkl'

os.makedirs('data/raw',       exist_ok=True)
os.makedirs('data/processed', exist_ok=True)

print('Config ready.')
print(f'  TIF source : {TIF_IN}')
print(f'  Delhi bbox : ({MIN_LON}, {MIN_LAT}) to ({MAX_LON}, {MAX_LAT})')

Config ready.
  TIF source : data/raw/population.tif
  Delhi bbox : (76.8, 28.4) to (77.4, 28.9)


## 3. Peek at TIF header — zero pixels loaded into RAM

rasterio reads only metadata here. Safe even on a 750MB file.

In [5]:
with rasterio.open(TIF_IN) as src:
    print('=== TIF file info (header only — no pixels loaded) ===')
    print(f'  CRS              : {src.crs}')
    print(f'  Full dimensions  : {src.width} cols x {src.height} rows')
    print(f'  Pixel resolution : {src.res[0]:.6f} deg x {src.res[1]:.6f} deg')
    print(f'  Full bounds      : {src.bounds}')
    print(f'  NoData value     : {src.nodata}')
    print(f'  Data type        : {src.dtypes[0]}')

    full_mb = src.width * src.height * 4 / 1_000_000
    print(f'\n  Full India in RAM would use ~{full_mb:.0f} MB  <-- DO NOT READ FULLY')

    win   = from_bounds(MIN_LON, MIN_LAT, MAX_LON, MAX_LAT, src.transform)
    d_mb  = int(win.width) * int(win.height) * 4 / 1_000_000
    print(f'  Delhi window     : {int(win.width)} cols x {int(win.height)} rows  (~{d_mb:.1f} MB  <-- safe)')
    print(f'  Memory saving    : {full_mb/max(d_mb,0.01):.0f}x less data')

RasterioIOError: data/raw/population.tif: No such file or directory

## 4. Extract Delhi slice from the 750MB TIF — windowed read

This is the key technique that keeps RAM usage low.  
rasterio calculates which rows and columns correspond to the Delhi
bounding box and reads ONLY those pixels. The rest of the 750MB
file is never touched. Peak RAM during this step: under 50MB.

In [ ]:
def extract_window_to_tif(tif_in, tif_out, min_lon, min_lat, max_lon, max_lat):
    """
    Read only the bounding box pixels from a large TIF using
    rasterio windowed reading. Saves the small tile to tif_out.
    Never loads the full raster into memory.
    """
    with rasterio.open(tif_in) as src:

        # Compute which pixel rows/cols correspond to the Delhi bbox
        window = from_bounds(
            left=min_lon, bottom=min_lat,
            right=max_lon, top=max_lat,
            transform=src.transform
        )

        print('Reading Delhi window from disk...')
        data = src.read(1, window=window)   # <-- only Delhi pixels read
        print(f'  Array shape : {data.shape}  (rows x cols)')
        print(f'  RAM used    : ~{data.nbytes / 1_000_000:.1f} MB')

        delhi_transform = src.window_transform(window)

        profile = src.profile.copy()
        profile.update({
            'height'   : data.shape[0],
            'width'    : data.shape[1],
            'transform': delhi_transform,
            'compress' : 'lzw'
        })

        with rasterio.open(tif_out, 'w', **profile) as dst:
            dst.write(data, 1)

        nodata = src.nodata if src.nodata is not None else -99999
        valid  = data[(data != nodata) & (data > 0)]

        print(f'\nDelhi TIF saved to {tif_out}')
        print(f'  File size on disk : {os.path.getsize(tif_out)/1e6:.1f} MB')
        print(f'  Valid pixels      : {len(valid):,}')
        print(f'  Population range  : {valid.min():.1f} to {valid.max():.1f} per 100m pixel')
        print(f'  Total pop in bbox : {valid.sum():,.0f}')

    del data
    gc.collect()


extract_window_to_tif(TIF_IN, TIF_DELHI, MIN_LON, MIN_LAT, MAX_LON, MAX_LAT)
print('\nFrom this point we only use the small Delhi TIF. 750MB file not touched again.')

## 5. Reproject to EPSG:4326 if needed

WorldPop is almost always already WGS84 — this cell checks and skips if so.

In [ ]:
def ensure_wgs84(tif_path):
    with rasterio.open(tif_path) as src:
        epsg = src.crs.to_epsg()
    if epsg == 4326:
        print(f'CRS is already EPSG:4326 — no reprojection needed.')
        return tif_path
    print(f'CRS is EPSG:{epsg} — reprojecting to EPSG:4326...')
    out = tif_path.replace('.tif', '_wgs84.tif')
    with rasterio.open(tif_path) as src:
        t, w, h = calculate_default_transform(
            src.crs, 'EPSG:4326', src.width, src.height, *src.bounds)
        meta = src.meta.copy()
        meta.update({'crs': 'EPSG:4326', 'transform': t, 'width': w, 'height': h})
        with rasterio.open(out, 'w', **meta) as dst:
            reproject(rasterio.band(src, 1), rasterio.band(dst, 1),
                      src_crs=src.crs, dst_crs='EPSG:4326',
                      resampling=Resampling.bilinear)
    print(f'Reprojected saved to {out}')
    return out

WORKING_TIF = ensure_wgs84(TIF_DELHI)
print(f'Working TIF: {WORKING_TIF}')

## 6. Quick visual sanity check

You should see Delhi's dense urban core as a bright orange/red blob
in the centre. If the plot is all grey or blank, the bbox or TIF path is wrong.

In [ ]:
with rasterio.open(WORKING_TIF) as src:
    data   = src.read(1).astype(float)
    bounds = src.bounds
    nodata = src.nodata if src.nodata is not None else -99999

data[data == nodata] = np.nan
data[data <= 0]      = np.nan

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

p99 = np.nanpercentile(data, 99)
im = axes[0].imshow(
    data, cmap='YlOrRd',
    norm=mcolors.LogNorm(vmin=0.1, vmax=p99),
    extent=[bounds.left, bounds.right, bounds.bottom, bounds.top],
    aspect='auto'
)
plt.colorbar(im, ax=axes[0], label='Pop per 100m pixel (log scale)')
axes[0].set_title('WorldPop 100m resolution — Delhi window')
axes[0].set_xlabel('Longitude') ; axes[0].set_ylabel('Latitude')

vals = data[~np.isnan(data)]
axes[1].hist(vals[vals < np.percentile(vals, 98)], bins=60,
             color='#D85A30', edgecolor='white', linewidth=0.3)
axes[1].axvline(np.nanmedian(data), color='#534AB7', linestyle='--',
                label=f'Median={np.nanmedian(data):.1f}')
axes[1].set_title('Population per 100m pixel — distribution')
axes[1].set_xlabel('Population') ; axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.tight_layout()
plt.savefig('data/processed/worldpop_delhi_check.png', dpi=110, bbox_inches='tight')
plt.show()

del data ; gc.collect()
print('Saved to data/processed/worldpop_delhi_check.png')

## 7. Load the 500m grid from 01_grid_build.ipynb

In [ ]:
gdf = gpd.read_file(GRID_PATH)

if str(gdf.crs) != 'EPSG:4326':
    gdf = gdf.to_crs('EPSG:4326')
    print('Grid reprojected to EPSG:4326')

print(f'Grid loaded  : {len(gdf):,} cells')
print(f'CRS          : {gdf.crs}')
print(f'Columns      : {list(gdf.columns)}')
gdf.head(2)

## 8. Zonal statistics — aggregate 100m pixels into 500m cells

Each 500m cell contains roughly 25 WorldPop pixels (a 5x5 block).  
rasterstats streams through the raster in chunks — RAM stays low.  
Expected time: 30 seconds to 2 minutes on a standard laptop.

In [ ]:
print(f'Running zonal statistics across {len(gdf):,} grid cells...')
print('rasterstats streams the raster — peak RAM stays under ~200MB.')

stats = zonal_stats(
    vectors     = gdf.geometry,
    raster      = WORKING_TIF,
    stats       = ['sum', 'mean', 'max', 'std', 'count'],
    nodata      = -99999,
    all_touched = True
)

stats_df = pd.DataFrame(stats)
stats_df.columns = ['pop_sum', 'pop_mean_100m', 'pop_max_100m',
                    'pop_std_100m', 'pop_pixel_count']

print(f'\nZonal stats done.')
print(f'  Total cells                : {len(stats_df):,}')
print(f'  Cells with population > 0  : {(stats_df.pop_sum > 0).sum():,}')
print(f'  Cells with no data (NaN)   : {stats_df.pop_sum.isna().sum():,}')

stats_df.describe()

## 9. Build all population feature columns

These are the columns that flow into your ML feature matrix.

In [ ]:
CELL_AREA_KM2 = 0.25   # 500m x 500m = 0.25 km2

# Raw aggregated values
gdf['pop_sum']         = stats_df['pop_sum'].fillna(0).clip(lower=0)
gdf['pop_mean_100m']   = stats_df['pop_mean_100m'].fillna(0).clip(lower=0)
gdf['pop_max_100m']    = stats_df['pop_max_100m'].fillna(0).clip(lower=0)
gdf['pop_std_100m']    = stats_df['pop_std_100m'].fillna(0).clip(lower=0)
gdf['pop_pixel_count'] = stats_df['pop_pixel_count'].fillna(0)

# pop_density: people per km2 — the primary feature
gdf['pop_density'] = gdf['pop_sum'] / CELL_AREA_KM2

# pop_density_log: log(x+1) transform — reduces extreme skew from city-centre cells
# LightGBM works better with this than the raw density
gdf['pop_density_log'] = np.log1p(gdf['pop_density'])

# pop_concentration: how much denser is the peak 100m pixel vs the cell average
# high value = population squeezed into one corner of the 500m cell
gdf['pop_concentration'] = gdf['pop_max_100m'] / (gdf['pop_mean_100m'] + 1e-9)

# pop_heterogeneity: coefficient of variation — is the cell uniform or patchy?
gdf['pop_heterogeneity'] = gdf['pop_std_100m'] / (gdf['pop_mean_100m'] + 1e-9)

# pop_is_empty: 1 if no residents — industrial zone, water, forest
# these cells go to the Thompson Sampling cold-start path in the model
gdf['pop_is_empty'] = (gdf['pop_sum'] < 1.0).astype(int)

POP_FEATURE_COLS = [
    'pop_sum',
    'pop_density',           # primary feature — used directly in feature_engineering.py
    'pop_density_log',       # log version — what LightGBM actually trains on
    'pop_mean_100m',
    'pop_max_100m',
    'pop_std_100m',
    'pop_concentration',
    'pop_heterogeneity',
    'pop_is_empty'
]

print('Feature columns created:')
for col in POP_FEATURE_COLS:
    nz = (gdf[col] > 0).sum()
    print(f'  {col:<25}  non-zero: {nz:,}  '
          f'mean: {gdf[col].mean():.1f}  max: {gdf[col].max():.1f}')

print(f'\nEmpty cells (cold start flag) : {gdf.pop_is_empty.sum():,}')

## 10. NaN check — model will crash if NaNs reach the feature matrix

In [ ]:
print('NaN check per feature column:')
all_clean = True
for col in POP_FEATURE_COLS:
    nans = gdf[col].isna().sum()
    infs = np.isinf(gdf[col].values).sum()
    status = 'OK'
    if nans > 0:
        status = f'WARNING {nans} NaNs'
        all_clean = False
    if infs > 0:
        status += f'  WARNING {infs} Infs'
        all_clean = False
        gdf[col] = gdf[col].replace([np.inf, -np.inf], 0)
    print(f'  {col:<25} {status}')

# Final fillna safety net
gdf[POP_FEATURE_COLS] = gdf[POP_FEATURE_COLS].fillna(0)

if all_clean:
    print('\nAll feature columns are clean — no NaNs or Infs.')
else:
    print('\nIssues found and auto-fixed with fillna(0).')

## 11. Visualise population on the 500m grid

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

populated = gdf[gdf['pop_density'] > 0]
empty     = gdf[gdf['pop_density'] == 0]

populated.plot(
    column='pop_density', ax=axes[0], cmap='YlOrRd',
    scheme='quantiles', k=7, legend=True,
    legend_kwds={'title': 'People/km2', 'fmt': '{:.0f}'},
    edgecolor='none', alpha=0.9
)
empty.plot(ax=axes[0], color='#EEEEEE', edgecolor='none')
axes[0].set_title('Population density per 500m cell\n(WorldPop real data)', fontsize=11)
axes[0].set_xlabel('Longitude') ; axes[0].set_ylabel('Latitude')

populated.plot(
    column='pop_density_log', ax=axes[1], cmap='plasma',
    scheme='quantiles', k=7, legend=True,
    legend_kwds={'title': 'log(people/km2)', 'fmt': '{:.1f}'},
    edgecolor='none', alpha=0.9
)
empty.plot(ax=axes[1], color='#EEEEEE', edgecolor='none')
axes[1].set_title('Log population density\n(feature the ML model uses)', fontsize=11)
axes[1].set_xlabel('Longitude') ; axes[1].set_ylabel('Latitude')

plt.tight_layout()
plt.savefig('data/processed/population_grid_maps.png', dpi=110, bbox_inches='tight')
plt.show()
print('Saved to data/processed/population_grid_maps.png')

## 12. Merge real population into features.pkl if 02_features already ran

In [ ]:
FEATURES_PATH = 'data/processed/features.pkl'

if os.path.exists(FEATURES_PATH):
    print('features.pkl found — merging real WorldPop data in...')
    feat_gdf = pd.read_pickle(FEATURES_PATH)

    # Drop any stale synthetic pop columns
    old = [c for c in POP_FEATURE_COLS if c in feat_gdf.columns]
    if old:
        feat_gdf = feat_gdf.drop(columns=old)
        print(f'  Removed old columns: {old}')

    feat_gdf = feat_gdf.merge(
        gdf[['grid_id'] + POP_FEATURE_COLS], on='grid_id', how='left'
    )
    feat_gdf[POP_FEATURE_COLS] = feat_gdf[POP_FEATURE_COLS].fillna(0)
    feat_gdf.to_pickle(FEATURES_PATH)

    print(f'  Updated shape: {feat_gdf.shape}')
    print('  pop_density is now REAL WorldPop data.')
else:
    print('features.pkl not found yet — run 02_features.ipynb next.')
    print('It will load grid_with_population.pkl and use real pop_density automatically.')

## 13. Save outputs

In [ ]:
# Output 1: GeoDataFrame with all population columns attached
gdf.to_pickle(OUT_GRID_PKL)
print(f'Saved: {OUT_GRID_PKL}')
print(f'       {len(gdf):,} cells x {len(gdf.columns)} columns')

# Output 2: numpy feature array for direct concatenation into feature matrix
pop_array = gdf[POP_FEATURE_COLS].values.astype(np.float32)

bundle = {
    'features'      : pop_array,
    'feature_names' : POP_FEATURE_COLS,
    'grid_ids'      : gdf['grid_id'].values,
    'n_cells'       : len(gdf)
}
with open(OUT_FEAT_PKL, 'wb') as f:
    pickle.dump(bundle, f)

print(f'\nSaved: {OUT_FEAT_PKL}')
print(f'       array shape : {pop_array.shape}')
print(f'       NaN count   : {np.isnan(pop_array).sum()}')

del pop_array, stats_df, stats
gc.collect()
print('\nMemory freed.')

## 14. Final summary

In [ ]:
print('=' * 58)
print('  WORLDPOP EXTRACTION COMPLETE')
print('=' * 58)
print(f'  Source TIF (untouched)   : {TIF_IN}')
print(f'  Source TIF size          : {os.path.getsize(TIF_IN)/1e6:.0f} MB')
print(f'  Delhi TIF extracted      : {TIF_DELHI}')
print(f'  Delhi TIF size           : {os.path.getsize(TIF_DELHI)/1e6:.1f} MB')
print()
print(f'  Grid cells total         : {len(gdf):,}')
print(f'  Populated cells          : {(gdf.pop_sum > 0).sum():,}')
print(f'  Empty cells (cold start) : {gdf.pop_is_empty.sum():,}')
print()
print(f'  Delhi total population   : {gdf.pop_sum.sum():,.0f}')
print(f'  Mean density (pop/km2)   : {gdf.pop_density.mean():,.0f}')
print(f'  Peak density (pop/km2)   : {gdf.pop_density.max():,.0f}')
print()
print('  Files saved:')
print(f'    {OUT_GRID_PKL}')
print(f'    {OUT_FEAT_PKL}')
print(f'    {TIF_DELHI}')
print()
print('  Next step: open 02_features.ipynb')
print('  Load with:')
print('    gdf = pd.read_pickle("data/processed/grid_with_population.pkl")')
print('  pop_density is now REAL — not synthetic.')
print('=' * 58)

## 15. What to change in 02_features.ipynb

Open `02_features.ipynb` and make two changes:

**Change 1 — Load the grid from this notebook's output instead of grid.geojson:**

```python
# REPLACE:
gdf = gpd.read_file('data/processed/grid.geojson')

# WITH:
gdf = pd.read_pickle('data/processed/grid_with_population.pkl')
```

**Change 2 — In generate_synthetic_data(), do not overwrite pop_density:**

```python
# REMOVE this line (or comment it out):
gdf['pop_density'] = np.random.normal(5000, 2000, len(gdf)).clip(min=0)

# pop_density already exists from WorldPop — do not regenerate it.
# All other synthetic features stay as-is:
# income_index, competitor_count, road_density, transit_stops, warehouse_proximity_km
```

That is all. The rest of the pipeline — spatial lag computation,
SMOTE, GWR, LightGBM, SHAP — reads pop_density as a normal column
and works without any other changes.